In [ ]:
# Run once: install pyyaml if not already installed
!pip install pyyaml

In [19]:
#Finding where am I
import os
os.environ['USE_LOCALSTACK'] = '1'
os.environ['LOCALSTACK_URL'] = 'http://localhost:4566'
# os.chdir('/config/workspace/python/pythonawsproject')
# print(os.getcwd())

# Now import (after chdir so Python can find the network package)
from network.state import load_state
print(load_state())

{'key_pair_name': 'dev-key'}


In [20]:
#What's the plan yo!
import yaml

with open('config/network.yaml', 'r') as f:
    config = yaml.safe_load(f) #//putting all the values from load state to config

print(config['vpc'])
print(config['security_groups'])

{'name': 'dev-vpc', 'cidr': '10.0.0.0/16', 'region': 'ap-south-1'}
[{'name': 'dev-web-sg', 'description': 'Allow SSH, HTTP, HTTPS inbound', 'ingress': [{'protocol': 'tcp', 'port': 22, 'cidr': '0.0.0.0/0', 'description': 'SSH access'}, {'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}, {'protocol': 'tcp', 'port': 443, 'cidr': '0.0.0.0/0', 'description': 'HTTPs'}], 'egress': [{'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}]}]


In [21]:
##Now it can work
from network.vpc import create_vpc

vpc_id = create_vpc(config)
print(f"Result: {vpc_id}")

Created VPC: vpc-d35b35c2
VPC is now available
Tagged VPC with: ['Name', 'ManagedBy', 'Layer', 'Environment']
State saved
Result: vpc-d35b35c2


In [22]:
from network.subnets import create_subnet
##We are utting full config python file here for execution
result = create_subnet(config)
print(result)

[{'name': 'public-subnet-az1', 'cidr': '10.0.1.0/24', 'az': 'ap-south-1a'}, {'name': 'public-subnet-az2', 'cidr': '10.0.2.0/24', 'az': 'ap-south-1b'}, {'name': 'private-subnet-az1', 'cidr': '10.0.3.0/24', 'az': 'ap-south-1a'}, {'name': 'private-subnet-az2', 'cidr': '10.0.4.0/24', 'az': 'ap-south-1b'}]
{'Subnet': {'AvailabilityZoneId': 'aps1-az1', 'MapCustomerOwnedIpOnLaunch': False, 'OwnerId': '000000000000', 'AssignIpv6AddressOnCreation': False, 'Ipv6CidrBlockAssociationSet': [], 'Tags': [], 'SubnetArn': 'arn:aws:ec2:ap-south-1:000000000000:subnet/subnet-29832428', 'EnableDns64': False, 'SubnetId': 'subnet-29832428', 'State': 'available', 'VpcId': 'vpc-d35b35c2', 'CidrBlock': '10.0.1.0/24', 'AvailableIpAddressCount': 251, 'AvailabilityZone': 'ap-south-1a', 'DefaultForAz': False, 'MapPublicIpOnLaunch': False}, 'ResponseMetadata': {'RequestId': '0e5e1cde-d165-4b4c-93a3-30ea7fa16cb4', 'HTTPStatusCode': 200, 'HTTPHeaders': {'content-length': '862', 'content-type': 'application/xml', 'date

In [23]:
from network.internet_gateway  import creating_igw
from network.state import load_state, save_state
from network.aws_client import get_client

igw_id = creating_igw(config)
print(f"Result: {igw_id}")
# ec2 = get_client('ec2')
# response = ec2.describe_internet_gateways(
#     Filters=[{'Name': 'attachment.vpc-id', 'Values': [vpc_id]}]
# )

# print(vpc_id)

# response = ec2.describe_internet_gateways()
# print(response['InternetGateways'])

# igw_id = response['InternetGateways'][0]['InternetGatewayId']
# print(f"Found exsisting IGW: {igw_id}")

# ec2.attach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)
# print(f"Attached {igw_id} to {vpc_id}")

# ##Saving and loading of state file
# state = load_state()
# state['igw_id'] = igw_id
# save_state(state)
# print("Saved State")

# igw_id = creating_igw(config)
# print(f"Result: {igw_id}")

Created IGW: igw-1f2fce9e
Attachment IGW to VPC: vpc-d35b35c2
Tagged  IGW: dev-igw
Saved IGW to State
Result: igw-1f2fce9e


In [24]:
from network.route_tables import create_route_tables

result = create_route_tables(config)

Created public route table: rtb-c441db69
Added route: '0.0.0.0/0' rtb-c441db69
Public route table already exists: rtb-c441db69
Created private route table: rtb-20483da6
Saved private route table to state
  Associated private-subnet-az1 → private route table
  Associated private-subnet-az2 → private route table
Route tables done.


In [25]:
from network.nat_gateway import create_nat_gateway

result = create_nat_gateway(config)

Allocated Elastic IP: eipalloc-80c88bdfbf92435ea
Created NAT Gateway: nat-8146029d3e9d969a4
Waiting for NAT Gateway to become available (1-2 min)...
NAT Gateway is now available
Saved NAT Gateway to state
Added route: 0.0.0.0/0 → nat-8146029d3e9d969a4 in private route table
NAT Gateway done.


In [26]:
from network.state import load_state, save_state

state = load_state()
del state['nat_id']
del state['nat_eip_alloc_id']

In [27]:
import os
os.chdir('/workspaces/floci-boto-3-python')
print(os.listdir('network/'))

['ec2_instance.py', 'key_pair.py', 'state.py', 'main.py', '__pycache__', 'nat_gateway.py', 'security_groups.py', 'internet_gateway.py', 'subnets.py', '__init__.py', 'route_tables.py', 'aws_client.py', 'planner.py', 'vpc.py']


In [28]:
from network.security_groups import create_security_groups
create_security_groups(config)

Created Security Group: sg-96a3c641a786f4946 (dev-web-sg)
  Added 3 ingress rules
  Revoked default egress (allow-all) rule
  Tagged + saved: sg_dev-web-sg = sg-96a3c641a786f4946
Security Groups done.


{'key_pair_name': 'dev-key',
 'vpc_id': 'vpc-d35b35c2',
 'vpc_name': 'dev-vpc',
 'public-subnet-az1': 'subnet-29832428',
 'public-subnet-az2': 'subnet-4f512080',
 'private-subnet-az1': 'subnet-0e5698a9',
 'private-subnet-az2': 'subnet-b4077173',
 'igw_id': 'igw-1f2fce9e',
 'public_rt_id': 'rtb-c441db69',
 'public-subnet-az1_rt_assoc': 'rtbassoc-38df0232',
 'public-subnet-az2_rt_assoc': 'rtbassoc-0d42c436',
 'private_rt_id': 'rtb-20483da6',
 'private-subnet-az1_rt_assoc': 'rtbassoc-d63f80c1',
 'private-subnet-az2_rt_assoc': 'rtbassoc-a1e9b2f6',
 'nat_eip_alloc_id': 'eipalloc-80c88bdfbf92435ea',
 'nat_id': 'nat-8146029d3e9d969a4',
 'private_nat_route': True,
 'sg_dev-web-sg': 'sg-96a3c641a786f4946'}

In [29]:
# from network.key_pair import create_key_pair
# create_key_pair(config)

import importlib
import network.key_pair

importlib.reload(network.key_pair)
key_name = network.key_pair.create_key_pair(config)
print("Returned key:", key_name)

Key pair 'dev-key' already exists in state and at keys/dev-key.pem
Returned key: dev-key


In [30]:

from network.ec2_instance import create_ec2_instance
instance_id = create_ec2_instance(config)
print(f"Result: {instance_id}")

Launched EC2 instance: i-4018aacc954426b78
Waiting for instance to reach 'running' state...
Instance is now running.
Save EC@ instance to state: i-4018aacc954426b78
Result: i-4018aacc954426b78


In [31]:
from network.planner import run_planner
plan = run_planner()

🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ✅ vpc                  "dev-vpc" → EXISTS (vpc-d35b35c2...)
  ✅ subnet               "public-subnet-az1" → EXISTS (subnet-29832428...)
  ✅ subnet               "public-subnet-az2" → EXISTS (subnet-4f512080...)
  ✅ subnet               "private-subnet-az1" → EXISTS (subnet-0e5698a9...)
  ✅ subnet               "private-subnet-az2" → EXISTS (subnet-b4077173...)
  ✅ internet_gateway     "dev-igw" → EXISTS (igw-1f2fce9e...)
  ✅ nat_gateway          "dev-nat" → EXISTS (nat-8146029d3e9d969a...)
  ✅ route_table          "public-route-table" → EXISTS (rtb-c441db69...)
  ✅ route_table          "private-route-table" → EXISTS (rtb-20483da6...)
  ✅ security_group       "dev-web-sg" → EXISTS (sg-96a3c641a786f4946...)
  0 to create, 0 to delete, 10 unchanged

  ✅ Infrastructure matches desired state. Nothing to do.


In [32]:
from network.main import command_plan
command_plan(config)


 Scanning the  battlefiend that is config file
🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ✅ vpc                  "dev-vpc" → EXISTS (vpc-d35b35c2...)
  ✅ subnet               "public-subnet-az1" → EXISTS (subnet-29832428...)
  ✅ subnet               "public-subnet-az2" → EXISTS (subnet-4f512080...)
  ✅ subnet               "private-subnet-az1" → EXISTS (subnet-0e5698a9...)
  ✅ subnet               "private-subnet-az2" → EXISTS (subnet-b4077173...)
  ✅ internet_gateway     "dev-igw" → EXISTS (igw-1f2fce9e...)
  ✅ nat_gateway          "dev-nat" → EXISTS (nat-8146029d3e9d969a...)
  ✅ route_table          "public-route-table" → EXISTS (rtb-c441db69...)
  ✅ route_table          "private-route-table" → EXISTS (rtb-20483da6...)
  ✅ security_group       "dev-web-sg" → EXISTS (sg-96a3c641a786f4946...)
  0 to create, 0 to delete, 10 unchanged

  ✅ Infrastructure matches desired state. Nothing to do.


[{'type': 'vpc', 'name': 'dev-vpc', 'action': 'NO-OP', 'id': 'vpc-d35b35c2'},
 {'type': 'subnet',
  'name': 'public-subnet-az1',
  'action': 'NO-OP',
  'id': 'subnet-29832428'},
 {'type': 'subnet',
  'name': 'public-subnet-az2',
  'action': 'NO-OP',
  'id': 'subnet-4f512080'},
 {'type': 'subnet',
  'name': 'private-subnet-az1',
  'action': 'NO-OP',
  'id': 'subnet-0e5698a9'},
 {'type': 'subnet',
  'name': 'private-subnet-az2',
  'action': 'NO-OP',
  'id': 'subnet-b4077173'},
 {'type': 'internet_gateway',
  'name': 'dev-igw',
  'action': 'NO-OP',
  'id': 'igw-1f2fce9e'},
 {'type': 'nat_gateway',
  'name': 'dev-nat',
  'action': 'NO-OP',
  'id': 'nat-8146029d3e9d969a4'},
 {'type': 'route_table',
  'name': 'public-route-table',
  'action': 'NO-OP',
  'id': 'rtb-c441db69'},
 {'type': 'route_table',
  'name': 'private-route-table',
  'action': 'NO-OP',
  'id': 'rtb-20483da6'},
 {'type': 'security_group',
  'name': 'dev-web-sg',
  'action': 'NO-OP',
  'id': 'sg-96a3c641a786f4946'}]

In [18]:
from network.main import command_destroy
command_destroy(config)


 RETREAT ORDER - Demolishing infrastructure ...

====== Terminate EC2 Instance ======
. Waiting for i-cd42c688c859ff898 to terminate...
====== Delete Key Pair ======
===== Destroying NAT Gateway =====
  Waiting for NAT nat-73c53b7a31318b281 to die...
  NAT Gateway deleted.
===== Releasing Elastic IP =====
  EIP released. Thanks for choosing the service.
===== Disassociating Route Tables =====
  Unfollowing: public-subnet-az1_rt_assoc
  Unfollowing: public-subnet-az2_rt_assoc
  Unfollowing: private-subnet-az1_rt_assoc
  Unfollowing: private-subnet-az2_rt_assoc
===== Deleting Route Tables =====
  Shredding: rtb-dc54286a
  Shredding: rtb-2d68c9e6
===== Detaching + Deleting IGW =====
  IGW igw-19fc8886 unhooked from VPC.
  IGW igw-19fc8886 deleted.
  IGW gone. No more front door.
 Deleteing Security Group: sg-40aedfcc7697469e3 (sg_dev-web-sg)
===== Deleting Subnets =====
  Demolishing: subnet-07c518fe (public-subnet-az1)
  Demolishing: subnet-f110c006 (public-subnet-az2)
  Demolishing: su

In [17]:
command_plan(config)


 Scanning the  battlefiend that is config file
🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ✅ vpc                  "dev-vpc" → EXISTS (vpc-d8978c7d...)
  ✅ subnet               "public-subnet-az1" → EXISTS (subnet-07c518fe...)
  ✅ subnet               "public-subnet-az2" → EXISTS (subnet-f110c006...)
  ✅ subnet               "private-subnet-az1" → EXISTS (subnet-2541809b...)
  ✅ subnet               "private-subnet-az2" → EXISTS (subnet-137931b1...)
  ✅ internet_gateway     "dev-igw" → EXISTS (igw-19fc8886...)
  ✅ nat_gateway          "dev-nat" → EXISTS (nat-73c53b7a31318b28...)
  ✅ route_table          "public-route-table" → EXISTS (rtb-dc54286a...)
  ✅ route_table          "private-route-table" → EXISTS (rtb-2d68c9e6...)
  ✅ security_group       "dev-web-sg" → EXISTS (sg-40aedfcc7697469e3...)
  0 to create, 0 to delete, 10 unchanged

  ✅ Infrastructure matches desired state. Nothing to do.


[{'type': 'vpc', 'name': 'dev-vpc', 'action': 'NO-OP', 'id': 'vpc-d8978c7d'},
 {'type': 'subnet',
  'name': 'public-subnet-az1',
  'action': 'NO-OP',
  'id': 'subnet-07c518fe'},
 {'type': 'subnet',
  'name': 'public-subnet-az2',
  'action': 'NO-OP',
  'id': 'subnet-f110c006'},
 {'type': 'subnet',
  'name': 'private-subnet-az1',
  'action': 'NO-OP',
  'id': 'subnet-2541809b'},
 {'type': 'subnet',
  'name': 'private-subnet-az2',
  'action': 'NO-OP',
  'id': 'subnet-137931b1'},
 {'type': 'internet_gateway',
  'name': 'dev-igw',
  'action': 'NO-OP',
  'id': 'igw-19fc8886'},
 {'type': 'nat_gateway',
  'name': 'dev-nat',
  'action': 'NO-OP',
  'id': 'nat-73c53b7a31318b281'},
 {'type': 'route_table',
  'name': 'public-route-table',
  'action': 'NO-OP',
  'id': 'rtb-dc54286a'},
 {'type': 'route_table',
  'name': 'private-route-table',
  'action': 'NO-OP',
  'id': 'rtb-2d68c9e6'},
 {'type': 'security_group',
  'name': 'dev-web-sg',
  'action': 'NO-OP',
  'id': 'sg-40aedfcc7697469e3'}]